In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

In [3]:
X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.95, 0.05],
    flip_y=0.01,
    class_sep=1.0,
    random_state=42
)

feature_names = [f"Feature_{i}" for i in range(1, 7)]
feature_names

['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_6']

In [4]:
print("X : \n", X)
print("\n\nY : \n", y)

X : 
 [[ 0.35138508  2.18237679 -0.25587452 -0.38825488  0.89272353 -0.86843608]
 [-1.80030098  0.16907513  1.64072894 -1.88487847  2.47515116  0.23696596]
 [-0.66256708 -0.96590379 -1.34998605 -0.15177221  0.03406364 -1.14730323]
 ...
 [-0.1167886  -1.89013984  0.88158498  0.32872479 -1.06747808 -1.54610246]
 [-0.92710117  2.51902558 -0.28023938  1.7840505   0.83015097  2.61703656]
 [ 0.0063384   2.03802907  0.62529819  0.28650582  0.9022181   2.53640932]]


Y : 
 [1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 

In [5]:
print(X.shape)
print(y.shape)

(1000, 6)
(1000,)


In [6]:
df = pd.DataFrame(X, columns=feature_names)
df['Fraud'] = y

In [7]:
print("Dataset shape : ", df.shape)
print("Class Distribution : ")
print(df['Fraud'].value_counts())
print("\nFraud Percentage : ", round(df['Fraud'].mean() * 100, 2), "%")

Dataset shape :  (1000, 7)
Class Distribution : 
Fraud
0    946
1     54
Name: count, dtype: int64

Fraud Percentage :  5.4 %


In [8]:
X = df[feature_names]
y = df['Fraud']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print("\nSplit sizes")
print("Training : ", len(X_train))
print("Validation : ", len(X_val))
print("Test : ", len(X_test))


Split sizes
Training :  600
Validation :  200
Test :  200


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression" : LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Random Forest" : RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

for model in models.values():
    model.fit(X_train_scaled, y_train)

In [10]:
MISSED_FRAUD_COST = 5000
FALSE_ALARM_COST = 200

thresholds = np.arange(0.05, 0.96, 0.01)

results = []
best_model_name = None
best_threshold = None
best_validation_cost = float('inf')

for name, model in models.items():
    val_probabilities = model.predict_proba(X_val_scaled)[:, 1]

    default_predictions = (val_probabilities >= 0.50).astype(int)
    default_precision = precision_score(y_val, default_predictions, zero_division=0)
    default_recall = recall_score(y_val, default_predictions, zero_division=0)
    default_f1 = f1_score(y_val, default_predictions, zero_division=0)

    val_pr_auc = average_precision_score(y_val, val_probabilities)

    model_best_cost = float("inf")
    model_best_threshold = None

    for threshold in thresholds:
        predictions = (val_probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        total_cost = (fn * MISSED_FRAUD_COST + fp * FALSE_ALARM_COST)

        if total_cost < model_best_cost:
            model_best_cost = total_cost
            model_best_threshold = threshold
    
    results.append({
        "Model" : name,
        "Validation Precision @ 0.50" : default_precision,
        "Validation Recall @ 0.50" : default_recall,
        "Validation F1 @ 0.50" : default_f1,
        "Validation PR-AUC" : val_pr_auc,
        "Best Threshold" : model_best_threshold,
        "Minimum Validation Cost (₹)" : model_best_cost,
    })

    if model_best_cost < best_validation_cost:
        best_validation_cost = model_best_cost
        best_model_name = name
        best_threshold = threshold

result_df = pd.DataFrame(results)
print("\n VALIDATION MODEL COMPARISION")
print(result_df.round(3).to_string(index=False))

print("\nSelected Model : ", best_model_name)
print("Selected threshold : ", round(best_threshold, 2))
print("Validation cost (₹) : ", round(best_validation_cost, 2))


 VALIDATION MODEL COMPARISION
              Model  Validation Precision @ 0.50  Validation Recall @ 0.50  Validation F1 @ 0.50  Validation PR-AUC  Best Threshold  Minimum Validation Cost (₹)
Logistic Regression                        0.094                     0.462                 0.156              0.229            0.06                        36200
      Random Forest                        0.750                     0.462                 0.571              0.601            0.17                        19200

Selected Model :  Random Forest
Selected threshold :  0.95
Validation cost (₹) :  19200


In [11]:
final_model = models[best_model_name]

test_probabilities = final_model.predict_proba(X_test_scaled)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
            y_test, test_predictions, labels=[0, 1]
        ).ravel()

test_precision = precision_score(y_test, test_predictions, zero_division=0)
test_recall = recall_score(y_test, test_predictions, zero_division=0)
test_f1 = f1_score(y_test, test_predictions, zero_division=0)
test_pr_auc = average_precision_score(y_test, test_probabilities)
test_business_cost = (fn * MISSED_FRAUD_COST + fp * FALSE_ALARM_COST)

print("Selected Model : ", best_model_name)
print("Threshold : ", round(best_threshold, 2))
print("Precision : ", round(test_precision, 2))
print("Recall : ", round(test_recall, 2))
print("F1-Score : ", round(test_f1, 2))
print("PR-AUC : ", round(test_pr_auc, 2))

print("\nConfusion Matrix : ")
print(np.array([[tn, fp], [fn, tp]]))

print("Estimated test business cost : ₹", test_business_cost)
print("Missed Fraud cost : ₹", fn * MISSED_FRAUD_COST)
print("False Alaram cost : ₹", fp * FALSE_ALARM_COST)

total_fraud_cases = tp + fn
print(f"Actual Fraud Cases In Test Set : {total_fraud_cases}")
print(f"Fraud cases detected : {tp}")
print(f"Fraud cases missed : {fn}")
print(f"Legitimate cases incorrectly flagged : {fp}")
print(f"Estimated total error cost : ₹{test_business_cost}")

Selected Model :  Random Forest
Threshold :  0.95
Precision :  0.0
Recall :  0.0
F1-Score :  0.0
PR-AUC :  0.6

Confusion Matrix : 
[[187   0]
 [ 13   0]]
Estimated test business cost : ₹ 65000
Missed Fraud cost : ₹ 65000
False Alaram cost : ₹ 0
Actual Fraud Cases In Test Set : 13
Fraud cases detected : 0
Fraud cases missed : 13
Legitimate cases incorrectly flagged : 0
Estimated total error cost : ₹65000
